In [ ]:
from datetime import datetime  # noqa: F401

import numpy as np  # noqa: F401
import pandas as pd
import pytz  # noqa: F401
import trimesh
from pv_index_for_run import pv_perf_index
from read_mesh import facenormal_angle, powerratio_iam, shadowing_factor, sky_face
from sun import get_sun_direction

Read out the car

In [ ]:
mesh = trimesh.load_mesh(
    "data/stl_models/outer_silvretta_shells/design_iter1_outer.stl"
)

read out thesampled points

In [ ]:
# --- about point.csv ---
# a file with points sampled in 15-minutes intervall.
# the car starts in Darwin at 10:00 local time, and travels
# perfectly south, at 85kph, for 8 hours.
df = pd.read_csv("data/race_routes/points.csv")

# convert to UTC
df["time"] = pd.to_datetime(df["time"], format="%Y-%m-%d %H:%M:%S")
df_time_acst = df["time"].dt.tz_localize("Australia/Darwin", ambiguous="infer")
df["time_utc"] = df_time_acst.dt.tz_convert("UTC")

# insight
df.head()

**Table/Matrix with corresponing performance values** 
- rows: Facettes
- columns: point in time

This table contains the weight of each facette of the mesh for every point in time. It could forexamplke be used to identy rows/facettes with the best correlation, (areas that perform and behave equally) this might be interesting to create a layout lateron.


In [ ]:
res = pd.DataFrame()
for index, row in df.iterrows():
    sun_dir = get_sun_direction(
        row["Lat"], row["Long"], row["altitude"], row["time_utc"]
    )
    top_mesh = sky_face(mesh)
    angles = facenormal_angle(top_mesh, sun_dir)
    shadow_mask = shadowing_factor(top_mesh, -1 * sun_dir)
    power_weight = powerratio_iam(angles)
    power_weight[shadow_mask] = 0
    res[f"{index}"] = power_weight

# res.to_csv('out.csv') #consider a faster dataformat for further use.  # noqa: ERA001

In [ ]:
# Computes the performance of the solar deck over the entire run
performance_index, selected_faces = pv_perf_index(top_mesh, res)
print("Performance index:", performance_index)
print("used area:", round((sum(selected_faces.facets_area) / 10e5), 3), "m2")

In [ ]:
selected_faces.visual.face_colors = [255, 0, 0, 150]

top_mesh.visual.face_colors = [0, 255, 0, 150]

scene = trimesh.Scene([top_mesh, selected_faces])
scene.show(notebook=True)